# Classical Abel–Jacobi MNIST training + accuracy

This notebook trains and evaluates the same MNIST models as `AJ_training_genus30.ipynb` and `scripts/aj_mnist_test_accuracy.py`:
- **Forward:** classical AJ axis-periodic model (image → AJ map → periodic features → classifier).
- **Inverse:** classical AJ P-function model (image → inverse AJ → classifier).

First it can train the forward AJ network (using precomputed genus-30 tables) and the inverse AJ network on MNIST, optionally saving checkpoints.
Then it can run the shared accuracy script and display the test accuracy and loss in a table and bar chart.

## Config

Set paths for data and optional checkpoints. Use `TEST_SUBSET > 0` for a quick run. Run from repo root or from this `notebooks/` folder.

In [1]:
import os
import re
import subprocess
import sys
from pathlib import Path

# Ensure repo root is importable, then use src.util to resolve paths.
for p in [Path.cwd(), Path.cwd().parent]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from src import util as aj_util

REPO_ROOT = aj_util.get_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_ROOT = REPO_ROOT / "data"
TABLES_DIR = ""  # optional; if empty the loader searches under DATA_ROOT and can auto-build tables
FORWARD_CKPT = ""  # optional path to forward checkpoint
INVERSE_CKPT = ""  # optional path to inverse checkpoint
TEST_SUBSET = 0    # 0 = full test set; >0 = use first N samples for quick run

## Train forward AJ axis-periodic network

This section trains the classical AJ axis-periodic MNIST model using genus-30 AJ lookup tables.
`TABLES_DIR` is optional: if provided, the loader tries that path first; otherwise it searches common locations under `DATA_ROOT`.
If no tables are found and auto-build is enabled, it computes and saves them automatically, then continues training.

In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from types import SimpleNamespace

from aj.classical import compute_aj_normalization

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device for forward AJ training:", device)


# ------------------------------------------------------------------
# Local replacements for the removed aj_test module
# ------------------------------------------------------------------
def _safe_torch_load(path, map_location=None):
    return aj_util.safe_torch_load(path, map_location=map_location)

def ensure_mnist_available(root):
    return aj_util.ensure_mnist_available(root)

def load_forward_tables(tables_dir, device):
    return aj_util.load_forward_tables(tables_dir, device)

def build_forward_model(tables_data, device, embed_dim=4, K=2):
    return aj_util.build_forward_model(tables_data, device, embed_dim=embed_dim, K=K)

def eval_epoch(model, loader, device):
    return aj_util.eval_epoch(model, loader, device)

# Backward-compatible namespace if older lines still reference aj_test
aj_test = SimpleNamespace(
    _safe_torch_load=_safe_torch_load,
    ensure_mnist_available=ensure_mnist_available,
    load_forward_tables=load_forward_tables,
    build_forward_model=build_forward_model,
    eval_epoch=eval_epoch,
)

FORWARD_EPOCHS = 5
FORWARD_LR = 3e-4
FORWARD_WEIGHT_DECAY = 1e-4
FORWARD_TRAIN_SUBSET = 20000   # 0 = full train set
FORWARD_BATCH_SIZE = 64
FORWARD_TEST_BATCH_SIZE = 256
FORWARD_CKPT_OUT = REPO_ROOT / "checkpoints" / "aj_forward_mnist_genus30.pt"

# Ensure MNIST is present
ensure_mnist_available(str(DATA_ROOT))

# Data loaders
train_loader_full, test_loader_fwd = aj_util.get_mnist_loaders(
    root=str(DATA_ROOT),
    test_batch_size=FORWARD_TEST_BATCH_SIZE,
    num_workers=0,
)
train_ds = train_loader_full.dataset
if FORWARD_TRAIN_SUBSET > 0:
    n = min(FORWARD_TRAIN_SUBSET, len(train_ds))
    train_ds = torch.utils.data.Subset(train_ds, list(range(n)))
    print(f"Forward AJ: using train subset of {n} samples")
else:
    print(f"Forward AJ: using full train set of {len(train_ds)} samples")

train_loader_fwd = torch.utils.data.DataLoader(
    train_ds, batch_size=FORWARD_BATCH_SIZE, shuffle=True, num_workers=0,
)

TABLES_GENUS = 30
TABLES_AUTO_BUILD = True
TABLES_GRID_SIZE = 96

tables, tables_found_dir = aj_util.get_or_build_forward_tables(
    device=device,
    tables_dir=(TABLES_DIR if TABLES_DIR else None),
    data_root=DATA_ROOT,
    genus=TABLES_GENUS,
    auto_build=TABLES_AUTO_BUILD,
    grid_size=TABLES_GRID_SIZE,
)
print(f"Using forward AJ tables from: {tables_found_dir}")

# Use aj.classical normalization directly.
mu_t, sigma_t = compute_aj_normalization(tables["I_plus"])
tables["mu_t"] = mu_t.float().to(device)
tables["sigma_t"] = sigma_t.float().to(device)

model_fwd = build_forward_model(tables, device)
opt_fwd = torch.optim.AdamW(model_fwd.parameters(), lr=FORWARD_LR, weight_decay=FORWARD_WEIGHT_DECAY)
ce = nn.CrossEntropyLoss()

print("Starting forward AJ training...")
for ep in range(1, FORWARD_EPOCHS + 1):
    model_fwd.train()
    tot, correct, n = 0.0, 0, 0
    for x, y in train_loader_fwd:
        x, y = x.to(device), y.to(device)
        opt_fwd.zero_grad(set_to_none=True)
        logits, aux = model_fwd(x, return_aux=True)
        loss = ce(logits, y)
        if aux is not None:
            loss = loss + 1e-3 * aux.get("branch_penalty", 0.0) + 1e-3 * aux.get("bound_penalty", 0.0)
        loss.backward()
        nn.utils.clip_grad_norm_(model_fwd.parameters(), 1.0)
        opt_fwd.step()
        tot += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        n += x.size(0)
    tr_loss, tr_acc = tot / n, 100.0 * correct / n
    te_loss, te_acc = eval_epoch(model_fwd, test_loader_fwd, device)
    print(f"[Forward AJ] Epoch {ep:02d} | train {tr_loss:.4f}/{tr_acc:.2f}% | test {te_loss:.4f}/{te_acc:.2f}%")

FORWARD_CKPT_OUT.parent.mkdir(parents=True, exist_ok=True)
torch.save({"state_dict": model_fwd.state_dict()}, FORWARD_CKPT_OUT)
FORWARD_CKPT = str(FORWARD_CKPT_OUT)
print("Saved forward AJ checkpoint to", FORWARD_CKPT)

Using device for forward AJ training: cuda
Forward AJ: using train subset of 20000 samples
Building forward AJ tables: genus=30, grid=96x96 (this may take a while)


KeyboardInterrupt: 

## Train inverse AJ P-network

This section trains the inverse Abel–Jacobi P-function MNIST model (`MNISTInversePNet`) using the same MNIST preprocessing.
It wraps the `scripts/train_inverse_p_mnist.py` utilities so that you can train and optionally save a checkpoint from this notebook.

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T

import scripts.train_inverse_p_mnist as inv_train

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device for inverse AJ training:", device)

INVERSE_GENUS = 2
INVERSE_EPOCHS = 5
INVERSE_BATCH_SIZE = 64
INVERSE_TRAIN_SUBSET = 20000   # 0 = full train set
INVERSE_TEST_SUBSET = 0        # 0 = full test set
INVERSE_CKPT_OUT = REPO_ROOT / "checkpoints" / f"mnist_inverse_p_g{INVERSE_GENUS}.pt"

tfm = T.Compose([T.ToTensor(), T.Normalize((0.1307,), (0.3081,))])
train_ds_inv = torchvision.datasets.MNIST(root=str(DATA_ROOT), train=True, download=True, transform=tfm)
test_ds_inv  = torchvision.datasets.MNIST(root=str(DATA_ROOT), train=False, download=True, transform=tfm)

if INVERSE_TRAIN_SUBSET > 0:
    n = min(INVERSE_TRAIN_SUBSET, len(train_ds_inv))
    train_ds_inv = torch.utils.data.Subset(train_ds_inv, list(range(n)))
    print(f"Inverse AJ: using train subset of {n} samples")
else:
    print(f"Inverse AJ: using full train set of {len(train_ds_inv)} samples")

if INVERSE_TEST_SUBSET > 0:
    m = min(INVERSE_TEST_SUBSET, len(test_ds_inv))
    test_ds_inv = torch.utils.data.Subset(test_ds_inv, list(range(m)))
    print(f"Inverse AJ: using test subset of {m} samples")
else:
    print(f"Inverse AJ: using full test set of {len(test_ds_inv)} samples")

train_loader_inv = torch.utils.data.DataLoader(
    train_ds_inv, batch_size=INVERSE_BATCH_SIZE, shuffle=True, num_workers=0,
)
test_loader_inv = torch.utils.data.DataLoader(
    test_ds_inv, batch_size=max(INVERSE_BATCH_SIZE, 128), shuffle=False, num_workers=0,
)

model_inv = inv_train.MNISTInversePNet(INVERSE_GENUS, device).to(device)
opt_inv = torch.optim.AdamW(model_inv.parameters(), lr=3e-4, weight_decay=1e-4)

print(f"Training MNIST inverse P model: genus={INVERSE_GENUS}, epochs={INVERSE_EPOCHS}")
for ep in range(1, INVERSE_EPOCHS + 1):
    tr_loss, tr_acc = inv_train.train_epoch(model_inv, train_loader_inv, opt_inv, device)
    te_loss, te_acc = inv_train.eval_epoch(model_inv, test_loader_inv, device)
    print(f"[Inverse AJ g={INVERSE_GENUS}] Epoch {ep:02d} | train {tr_loss:.4f}/{tr_acc:.2f}% | test {te_loss:.4f}/{te_acc:.2f}%")

INVERSE_CKPT_OUT.parent.mkdir(parents=True, exist_ok=True)
torch.save({"state_dict": model_inv.state_dict()}, INVERSE_CKPT_OUT)
INVERSE_CKPT = str(INVERSE_CKPT_OUT)
print("Saved inverse AJ checkpoint to", INVERSE_CKPT)